In [5]:
import torch
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
print("gpu name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("using:", device)


cuda available: True
device count: 1
gpu name: Tesla T4
using: cuda


In [ ]:
from pathlib import Path
from PIL import Image

root = Path("/root/.cache/kagglehub/datasets/odins0n/ucf-crime-dataset/versions/1")

all_png = sorted(root.rglob("*.png"))
print("total png:", len(all_png))

clips = {}
for p in all_png:
    clips.setdefault(p.parent, []).append(p)

print("total clips:", len(clips))

first_clip_dir, first_frames = next(iter(clips.items()))
first_frames = sorted(first_frames)
print("first clip dir:", first_clip_dir)
print("num frames:", len(first_frames))
print("first 5 frames:")
for p in first_frames[:5]:
    print(p)

img = Image.open(first_frames[0]).convert("RGB")
print("image size:", img.size)

total png: 1377653
total clips: 28
first clip dir: /root/.cache/kagglehub/datasets/odins0n/ucf-crime-dataset/versions/1/Test/Abuse
num frames: 297
first 5 frames:
/root/.cache/kagglehub/datasets/odins0n/ucf-crime-dataset/versions/1/Test/Abuse/Abuse028_x264_0.png
/root/.cache/kagglehub/datasets/odins0n/ucf-crime-dataset/versions/1/Test/Abuse/Abuse028_x264_10.png
/root/.cache/kagglehub/datasets/odins0n/ucf-crime-dataset/versions/1/Test/Abuse/Abuse028_x264_100.png
/root/.cache/kagglehub/datasets/odins0n/ucf-crime-dataset/versions/1/Test/Abuse/Abuse028_x264_1000.png
/root/.cache/kagglehub/datasets/odins0n/ucf-crime-dataset/versions/1/Test/Abuse/Abuse028_x264_1010.png
image size: (64, 64)


In [ ]:
import re
from collections import defaultdict

video_clips = defaultdict(list)
pat = re.compile(r"^(.*)_\d+$")  

for p in all_png:
    stem = p.stem
    m = pat.match(stem)
    video_id = m.group(1) if m else stem
    key = (p.parent.parent.name, p.parent.name, video_id)  
    video_clips[key].append(p)

for k in video_clips:
    video_clips[k] = sorted(video_clips[k], key=lambda x: int(x.stem.split("_")[-1]))

print("total videos:", len(video_clips))
print("sample key:", next(iter(video_clips.keys())))
print("sample frames:", len(next(iter(video_clips.values()))))


total videos: 1900
sample key: ('Test', 'Abuse', 'Abuse028_x264')
sample frames: 142


In [26]:
def y_from_class(class_name: str):
    c = class_name.strip().lower()
    return 0 if "normal" in c else 1



In [ ]:
#!pip -q install open_clip_torch ftfy regex tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00


In [ ]:
import random
import numpy as np
import torch
import open_clip
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# 1) 采样
MAX_PER_CLASS = 30
FRAMES_PER_VIDEO = 8
random.seed(42)

normal_keys = [k for k in video_clips if k[1].lower() in ["normal", "normalvideos", "normal_video"]]
anom_keys = [k for k in video_clips if k not in normal_keys]

print("normal_keys:", len(normal_keys), "anom_keys:", len(anom_keys))
assert len(normal_keys) > 0, "No normal class found. Check class names in k[1]."
assert len(anom_keys) > 0, "No anomaly class found."

n = min(len(normal_keys), len(anom_keys), 30)
sample_keys = random.sample(normal_keys, n) + random.sample(anom_keys, n)
random.shuffle(sample_keys)
print("normal/anomaly:", n, n)

normal_keys: 950 anom_keys: 950
normal/anomaly: 30 30


In [22]:
# 2) 加载CLIP
device = "cuda" if torch.cuda.is_available() else "cpu"
model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
tokenizer = open_clip.get_tokenizer("ViT-B-32")
model = model.to(device).eval()

prompts = ["a normal surveillance video", "a criminal or abnormal event in surveillance video"]
with torch.no_grad():
    text_tokens = tokenizer(prompts).to(device)
    text_features = model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)


/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


In [23]:
# 3) 推理函数
def pick_frames(paths, k=8):
    n = len(paths)
    idx = np.linspace(0, n - 1, num=min(k, n), dtype=int)
    return [paths[i] for i in idx]

@torch.no_grad()
def predict_one_video(frame_paths):
    chosen = pick_frames(frame_paths, FRAMES_PER_VIDEO)
    imgs = torch.stack([preprocess(Image.open(p).convert("RGB")) for p in chosen]).to(device)

    img_feat = model.encode_image(imgs)
    img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
    vid_feat = img_feat.mean(dim=0, keepdim=True)
    vid_feat = vid_feat / vid_feat.norm(dim=-1, keepdim=True)

    probs = (100.0 * vid_feat @ text_features.T).softmax(dim=-1).squeeze(0)
    pred = int(torch.argmax(probs).item())  # 0 normal, 1 anomaly
    return pred, probs.cpu().numpy()


In [27]:
# 4) 主循环 + 指标
y_true, y_pred = [], []

for key in tqdm(sample_keys):
    split_name, class_name, video_id = key
    frames = video_clips[key]
    true_y = y_from_class(class_name)

    pred_y, _ = predict_one_video(frames)
    y_true.append(true_y)
    y_pred.append(pred_y)

print("Accuracy:", accuracy_score(y_true, y_pred))
print("F1:", f1_score(y_true, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=["normal", "anomaly"]))


100%|██████████| 60/60 [00:02<00:00, 23.35it/s]

Accuracy: 0.4
F1: 0.21739130434782608
Confusion matrix:
 [[19 11]
 [25  5]]
              precision    recall  f1-score   support

      normal       0.43      0.63      0.51        30
     anomaly       0.31      0.17      0.22        30

    accuracy                           0.40        60
   macro avg       0.37      0.40      0.37        60
weighted avg       0.37      0.40      0.37        60



In [28]:
from collections import Counter
print("y_true dist:", Counter(y_true))
print("sample class dist:", Counter([k[1] for k in sample_keys]))


y_true dist: Counter({1: 30, 0: 30})
sample class dist: Counter({'NormalVideos': 30, 'Stealing': 5, 'Burglary': 5, 'Shooting': 4, 'Explosion': 3, 'Arrest': 3, 'RoadAccidents': 2, 'Abuse': 2, 'Robbery': 2, 'Fighting': 1, 'Shoplifting': 1, 'Arson': 1, 'Assault': 1})


The method uses a binary setup: normal vs anomaly.
Any class containing “normal” is labeled as normal; all other classes are grouped into anomaly.

The current notebook result is weak: Accuracy = 40%, F1 = 0.217.
The main issue is poor anomaly detection: only 5 out of 30 anomaly videos were correctly identified (about 17% recall), with 25 missed.
Normal videos perform somewhat better: 19 out of 30 were correctly classified.
This suggests the zero-shot CLIP setup is biased toward predicting “normal” and is not sensitive enough to anomalous events.